https://formal.org/

https://electronics.stackexchange.com/questions/331393/what-is-the-difference-between-logic-reg-and-wire-in-system-verilog-explainati
https://electronics.stackexchange.com/questions/99223/relation-between-delta-cycle-and-event-scheduling-in-verilog-simulation


 https://dl.acm.org/doi/10.1145/3763084 Revamping Verilog Semantics for Foundational Verification

system verilog spec https://rfsoc.mit.edu/6S965/_static/F25/documentation/1800-2017.pdf. 

Clifford Cummings. 2000. Nonblocking Assignments in Verilog Synthesis, Coding Styles That Kill! SNUG (SynopsysUsers Group) 2000 User Papers (2000)

https://ieeexplore.ieee.org/document/523251 The semantic challenge of Verilog HDL gordon
This SR latch thing  is kind of crazy.
A single timestep can get into an infinite loop in verilog?
I didn't know about dassignement with explicit delay



https://prod-ms-be.lib.mcmaster.ca/server/api/core/bitstreams/110f325f-0ff7-43b6-a5e2-bcbcdf88166f/content bluespec thesis pvs





A disjoint union of action rules? Is it important to have a selector saying which rule fires?


FRP as a touchstone?
delta cycles
time = (int, int) ? like pcode. like timely dataflow
[[count = 0, count = 1], [count = 1, count = 2], ...]
LTS labelled transitions system

Noah talked about bluespec.


fifo
priority arbiter bug
throttler


req signals, combinatorial output. Pick on of the request signas
$clog2
$onehot


Surely I must have a file I'm working with somewhere

What about just raw ingesting verilog

Amaranth producing. But like what do i even want to build in verilog

VTR / VPR is like an alntertae universe to nextpnr. Slow, simulated annealing config file based.
Still I'd like to see those files


https://github.com/StefanSchippers/xschem


Sets of events (mulitset?)
Event -> Set Event
Event is timestamped

datalog mtl


Multi-time verilog.
Multi time scale physics. Asymptotic expansions
Infintesimals 1 + 4dt or hyperreals for time?

The is "parallel" GR time. out of sync systems
But also hierarchical time, each time step of outer system may have artbirary time steps of inner system.

https://en.wikipedia.org/wiki/Discrete-event_simulation

https://arxiv.org/abs/2502.19348 simulation semantics of syntehsizable verilog
https://andreasloow.github.io/vv/

In [ ]:
import cocotb



In [ ]:
import heapq

sched = []
state = {}
response = {} # sensititivity list

from dataclasses import dataclass, field
from typing import Any

@dataclass
class State:
    data : dict[str, object] = field(default_factory=dict)
    listeners : dict[str, object] = field(default_factory=dict)
    def __setitem__(self, key, value):
        self.data[key] = value
        for f in self.listeners.get(key, []):
            f(self)
    def __getitem__(self, key):
        return self.data[key]

@dataclass(order=True)
class Event:
    time : int # tuple[int, int] ?
    item: Any=field(compare=False)


st = State()
while sched:
    ev = heapq.heappop(sched)
    ev.item(state)







coroutine based?
https://docs.cocotb.org/en/v1.8.0/coroutines.html

What would it look like to do this raw?



In [50]:
%%file /tmp/sr.sv
module foo;

var logic s,r;
wire q,qbar;
assign #1 qbar = ~(s | q);
assign #1 q = ~(r | qbar);

initial begin
 $display("Hello, world!");
 $monitor("s=%b r=%b q=%b qbar=%b, t=%d", s, r, q, qbar, $time);
 # 10 s = 0;
 # 10 s = 1;
# 10 r = 0;
# 10 r = 1;
end

endmodule


Overwriting /tmp/sr.sv


In [51]:
! verilator --binary -j 0 /tmp/sr.sv && /tmp/sr

make: Entering directory '/home/philip/philzook58.github.io/_drafts/obj_dir'
g++ -Os  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=0 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=0 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-shadow -Wno-sign-compare -Wno-tautological-compare -Wno-uninitialized -Wno-unused-but-set-parameter -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable    -DVL_TIME_CONTEXT   -fcoroutines -c -o verilated.o /usr/local/share/verilator/include/verilated.cpp
g++ -Os  -I.  -MMD -I/usr/local/share/verilator/include -I/usr/local/share/verilator/include/vltstd -DVM_COVERAGE=0 -DVM_SC=0 -DVM_TRACE=0 -DVM_TRACE_FST=0 -DVM_TRACE_VCD=0 -faligned-new -fcf-protection=none -Wno-bool-operation -Wno-shadow -Wno-sign-compare -Wno-tautological-compare -Wno-uninitialized -Wno-unused-but-set-parameter -Wno-unused-but-set-variable -Wno-unused-parameter -Wno-unused-variable    -DVL_TI

In [52]:
! iverilog -o /tmp/sr -g2012 /tmp/sr.sv && /tmp/sr

Hello, world!
s=x r=x q=x qbar=x, t=                   0
s=0 r=x q=x qbar=x, t=                  10
s=1 r=x q=x qbar=x, t=                  20
s=1 r=x q=x qbar=0, t=                  21
s=1 r=0 q=x qbar=0, t=                  30
s=1 r=0 q=1 qbar=0, t=                  31
s=1 r=1 q=1 qbar=0, t=                  40
s=1 r=1 q=0 qbar=0, t=                  41


Interesting. Start as z and unknown values x

In [1]:
%%file /tmp/counter.v

module mycounter(
    input clk,
    input rst,
    output reg [4:0] v
);

    always @(posedge clk) begin
        if (rst)
            v <= 5'd0;
        else
            v <= v + 5'd1;
    end

endmodule

Writing /tmp/counter.v


In [23]:
from pyosys import libyosys as ys
#ys.run_pass("read_verilog /tmp/counter.v")
design = ys.Design()
ys.run_pass("read_verilog -sv /tmp/counter.v", design)

mod = design.top_module()
mod.name
mod.ports[0]
str(mod.ports[1])



-- Running command `read_verilog -sv /tmp/counter.v' --

20. Executing Verilog-2005 frontend: /tmp/counter.v
Parsing SystemVerilog input from `/tmp/counter.v' to AST representation.
Generating RTLIL representation for module `\mycounter'.
Successfully finished Verilog frontend.


'\\rst'

# Formal.org
super llm generated
Interesting little hardware subpieces though

request grant on bus arbiter
```


priorirty encoder.

```
req[0]   req[1]   gnt
  0        0       X     (no request; output don't-care)
  1        0       0     (only req[0])
  0        1       1     (only req[1])
  1        1       0     (req[0] is higher priority)
```


error accumulators and stall assumulators


fifo
input clk, rst, wr_en, rd_en, 
output full, overflow, count

4-to-1 multiplexer

$past operator


https://github.com/ARC-Lab-UF/sv-tutorial/tree/main

https://vhdlwhiz.com/delta-cycles-explained/

In [ ]:
%%file /tmp/test.v

module test(
    intput a,
    
)


In [15]:
%%file /tmp/irq.sv

`default_nettype none
module irq_aggregate (
  input  wire timer_irq,
  input  wire uart_irq,
  output wire cpu_irq
);
  assign cpu_irq = timer_irq | uart_irq;
endmodule

module formal_tb (
  input wire       clk,
  input wire       rst,
  input wire       timer_irq,
  input wire       uart_irq
);
  wire cpu_irq;
  irq_aggregate    dut(.timer_irq(timer_irq), .uart_irq(uart_irq), .cpu_irq(cpu_irq));
  properties       p1 (.clk(clk), .rst(rst), .timer_irq(timer_irq), .uart_irq(uart_irq), .cpu_irq(cpu_irq));
endmodule



`default_nettype none
module properties (
  input wire clk,
  input wire rst,
  input wire timer_irq,
  input wire uart_irq,
  input wire cpu_irq
);
  reg f_past_valid = 0;
  always_ff @(posedge clk) f_past_valid <= 1;
  always_ff @(posedge clk) begin
    if (!f_past_valid)
      assume (rst);
    else
      assume (!rst);
  end

always_ff @(posedge clk)
  if (f_past_valid && !rst)
    if (timer_irq) assert (cpu_irq);

/*
always_ff @(posedge clk)
  if (f_past_valid && !rst)
    if (cpu_irq) assert (timer_irq);
*/
endmodule

Overwriting /tmp/irq.sv


In [20]:
! ebmc /tmp/irq.sv  # seemingly unnecessary? --top formal_tb 

Converting
Type-checking Verilog::formal_tb
Synthesis Verilog::formal_tb
No engine given, attempting heuristic engine selection
Attempting tautology check
Attempting transition property
Using MiniSAT 2.2.1 with simplifier
Checking transition properties
Checking formal_tb.p1.assert.2 using transition property engine
SAT checker: instance is UNSATISFIABLE
UNSAT: property holds for all transitions

** Results:
[formal_tb.p1.0.assume.1] assume always properti...: ASSUMED
[formal_tb.p1.assert.2] always properties.cpu_ir...: PROVED (transition property)


In [17]:
%%file /tmp/prop.sby
[tasks]
bmc

[options]
mode bmc
depth 20

[engines]
smtbmc boolector

[script]
read -formal /tmp/irq.sv
prep -top formal_tb

[files]
/tmp/irq.sv

Overwriting /tmp/prop.sby


In [14]:
! ~/Downloads/oss-cad-suite-linux-x64-20260504/oss-cad-suite/bin/sby -f /tmp/prop.sby

SBY 13:01:52 [/tmp/prop_bmc] Removing directory '/tmp/prop_bmc'.
SBY 13:01:52 [/tmp/prop_bmc] Copy '/tmp/irq.sv' to '/tmp/prop_bmc/src/irq.sv'.
SBY 13:01:52 [/tmp/prop_bmc] engine_0: smtbmc boolector
SBY 13:01:52 [/tmp/prop_bmc] base: starting process "cd /tmp/prop_bmc/src; yosys -ql ../model/design.log ../model/design.ys"
SBY 13:01:52 [/tmp/prop_bmc] base: finished (returncode=0)
SBY 13:01:52 [/tmp/prop_bmc] prep: starting process "cd /tmp/prop_bmc/model; yosys -ql design_prep.log design_prep.ys"
SBY 13:01:52 [/tmp/prop_bmc] prep: finished (returncode=0)
SBY 13:01:52 [/tmp/prop_bmc] smt2: starting process "cd /tmp/prop_bmc/model; yosys -ql design_smt2.log design_smt2.ys"
SBY 13:01:52 [/tmp/prop_bmc] smt2: finished (returncode=0)
SBY 13:01:52 [/tmp/prop_bmc] engine_0: starting process "cd /tmp/prop_bmc; yosys-smtbmc -s boolector --presat --unroll --noprogress -t 20  --append 0 --dump-vcd engine_0/trace.vcd --dump-yw engine_0/trace.yw --dump-vlogtb engine_0/trace_tb.v --dump-smtc engine